# Telco Customer Churn — Data Preparation

This notebook focuses on preparing the raw Telco Customer Churn dataset for exploratory analysis and machine learning tasks.

Main goals:
- Inspect dataset structure and data quality
- Optimize data types and memory usage
- Handle missing values and inconsistent formats
- Prepare a clean dataset for downstream analysis

## Import section

In [1]:
import os
import pandas as pd

## Load Raw Dataset

Load the original customer churn dataset and inspect sample records.

In [2]:
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [3]:
df.sample(10)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
7002,9586-JGQKH,Female,0,Yes,No,64,Yes,Yes,Fiber optic,No,...,No,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),105.40,6794.75,No
6623,9248-OJYKK,Male,1,No,No,1,Yes,Yes,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,76.45,76.45,Yes
972,0556-FJEGU,Male,0,No,No,58,Yes,No,DSL,No,...,Yes,Yes,Yes,No,Two year,No,Bank transfer (automatic),70.10,4048.95,No
4180,5482-PLVPE,Female,1,No,No,2,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,24.35,41.85,Yes
2702,1942-OQFRW,Male,0,No,No,1,Yes,No,DSL,No,...,No,No,No,No,Month-to-month,No,Electronic check,44.00,44,No
5699,6719-OXYBR,Male,0,No,No,15,Yes,No,Fiber optic,No,...,Yes,No,Yes,No,Month-to-month,No,Electronic check,85.30,1219.85,No
2244,0866-QLSIR,Female,0,No,No,34,Yes,Yes,DSL,Yes,...,No,Yes,No,No,One year,Yes,Mailed check,64.40,2088.75,Yes
3205,3810-DVDQQ,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,Yes,...,Yes,Yes,Yes,Yes,Two year,Yes,Bank transfer (automatic),117.60,8308.9,No
6747,5245-VDBUR,Female,0,Yes,No,52,No,No phone service,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,35.45,1958.95,No
3728,6352-GIGGQ,Male,0,No,No,67,Yes,Yes,DSL,Yes,...,Yes,Yes,Yes,Yes,One year,Yes,Bank transfer (automatic),88.80,5903.15,No


## Initial Dataset Inspection

Analyze:
- column types
- memory usage
- descriptive statistics
- unique values
- missing values
- duplicated rows

In [4]:
# Inspect memory usage and current data types
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [5]:
# Analyze statistics summary for numerical columns
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [6]:
# Analyze feature cardinality to identify categorical candidates
df.nunique()

customerID          7043
gender                 2
SeniorCitizen          2
Partner                2
Dependents             2
tenure                73
PhoneService           2
MultipleLines          3
InternetService        3
OnlineSecurity         3
OnlineBackup           3
DeviceProtection       3
TechSupport            3
StreamingTV            3
StreamingMovies        3
Contract               3
PaperlessBilling       2
PaymentMethod          4
MonthlyCharges      1585
TotalCharges        6531
Churn                  2
dtype: int64

In [7]:
# Check missing values for fully columns
df.isnull().sum()

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [8]:
# Check for fully duplicated rows
df.duplicated().sum()

np.int64(0)

## Convert Low Cardinality Features to Categorical Types

Columns with a small number of unique values are converted to `category`
to improve memory efficiency and semantic representation.

In [9]:
to_category = [col for col in df.columns if df[col].nunique() <= 4]

# Convert low-cardinality features into categorical types
for col in to_category:
	df[col] = df[col].astype('category')

In [10]:
# float32 provides sufficient precision for monetary values in this dataset
df['MonthlyCharges'] = df['MonthlyCharges'].astype('float32')

# int16 is sufficient since tenure values range from 0 to 72
df['tenure'] = df['tenure'].astype('int16')

## Handle Numeric Conversion

`TotalCharges` is stored as a string column.
Convert it into numeric format and coerce non-convertible values into NaN for further inspection.

In [11]:
# Convert TotalCharges into numeric values
df['TotalCharges'] = pd.to_numeric(
	df['TotalCharges'], errors='coerce', downcast='float'
)

## Standardize Column Naming

Improve naming consistency and readability.

In [12]:
df.rename(
	columns={'gender': 'Gender', 'tenure': 'Tenure'},
	inplace=True,
)

## Remove Non-Predictive Features

`customerID` uniquely identifies customers but does not provide predictive value for churn modeling.

In [13]:
df.drop(columns='customerID', inplace=True)

df_to_modelling = df.drop(columns='TotalCharges')

## Validate Dataset After Transformations

Re-evaluate:
- memory usage
- missing values
- duplicated rows
- dataset consistency

In [14]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   Gender            7043 non-null   category
 1   SeniorCitizen     7043 non-null   category
 2   Partner           7043 non-null   category
 3   Dependents        7043 non-null   category
 4   Tenure            7043 non-null   int16   
 5   PhoneService      7043 non-null   category
 6   MultipleLines     7043 non-null   category
 7   InternetService   7043 non-null   category
 8   OnlineSecurity    7043 non-null   category
 9   OnlineBackup      7043 non-null   category
 10  DeviceProtection  7043 non-null   category
 11  TechSupport       7043 non-null   category
 12  StreamingTV       7043 non-null   category
 13  StreamingMovies   7043 non-null   category
 14  Contract          7043 non-null   category
 15  PaperlessBilling  7043 non-null   category
 16  PaymentMethod     7043 non-null   c

In [15]:
df.describe()

,Tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7032.000000
mean,32.371149,64.761696,2283.300537
std,24.559481,30.090048,2266.771484
min,0.000000,18.250000,18.799999
25%,9.000000,35.500000,401.450012
50%,29.000000,70.349998,1397.475098
75%,55.000000,89.849998,3794.737549
max,72.000000,118.750000,8684.799805


In [16]:
df.isnull().sum()

Gender               0
SeniorCitizen        0
Partner              0
Dependents           0
Tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

### Duplicate Records Analysis

After removing `customerID`, some rows may appear duplicated.
These records do not necessarily indicate data quality problems, since different customers can share identical observable attributes.

In [17]:
df.duplicated().sum()

np.int64(22)

## Investigate Missing Values

Analyze rows containing missing values in `TotalCharges`.

In [18]:
# Preview all data with missing TotalCharges
df[df['TotalCharges'].isnull()]

,Gender,SeniorCitizen,Partner,Dependents,Tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
488,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,No,Yes,Yes,Yes,No,Two year,Yes,Bank transfer (automatic),52.549999,NaN,No
753,Male,0,No,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.250000,NaN,No
936,Female,0,Yes,Yes,0,Yes,No,DSL,Yes,Yes,Yes,No,Yes,Yes,Two year,No,Mailed check,80.849998,NaN,No
1082,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.750000,NaN,No
1340,Female,0,Yes,Yes,0,No,No phone service,DSL,Yes,Yes,Yes,Yes,Yes,No,Two year,No,Credit card (automatic),56.049999,NaN,No
3331,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,19.850000,NaN,No
3826,Male,0,Yes,Yes,0,Yes,Yes,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,25.350000,NaN,No
4380,Female,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,20.000000,NaN,No
5218,Male,0,Yes,Yes,0,Yes,No,No,No internet service,No internet service,No internet service,No internet service,No internet service,No internet service,One year,Yes,Mailed check,19.700001,NaN,No
6670,Female,0,Yes,Yes,0,Yes,Yes,DSL,No,Yes,Yes,Yes,Yes,No,Two year,No,Mailed check,73.349998,NaN,No


In [19]:
# Customers with missing TotalCharges typically have zero tenure
df['TotalCharges'] = df['TotalCharges'].fillna(0)

In [20]:
df.isnull().sum()

Gender              0
SeniorCitizen       0
Partner             0
Dependents          0
Tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

## Final Dataset Validation

Perform a final inspection before exporting the cleaned dataset.

In [21]:
df.info(memory_usage='deep')

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype   
---  ------            --------------  -----   
 0   Gender            7043 non-null   category
 1   SeniorCitizen     7043 non-null   category
 2   Partner           7043 non-null   category
 3   Dependents        7043 non-null   category
 4   Tenure            7043 non-null   int16   
 5   PhoneService      7043 non-null   category
 6   MultipleLines     7043 non-null   category
 7   InternetService   7043 non-null   category
 8   OnlineSecurity    7043 non-null   category
 9   OnlineBackup      7043 non-null   category
 10  DeviceProtection  7043 non-null   category
 11  TechSupport       7043 non-null   category
 12  StreamingTV       7043 non-null   category
 13  StreamingMovies   7043 non-null   category
 14  Contract          7043 non-null   category
 15  PaperlessBilling  7043 non-null   category
 16  PaymentMethod     7043 non-null   c

In [22]:
df.describe()

,Tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000
mean,32.371149,64.761696,2279.734375
std,24.559481,30.090048,2266.794434
min,0.000000,18.250000,0.000000
25%,9.000000,35.500000,398.549988
50%,29.000000,70.349998,1394.550049
75%,55.000000,89.849998,3786.599976
max,72.000000,118.750000,8684.799805


## Export Processed Dataset

Save the cleaned dataset in Parquet format to preserve schema and optimized data types.

In [23]:
os.makedirs('../data/processed/', exist_ok=True)
os.makedirs('../data/interim/', exist_ok=True)

In [24]:
# Export cleaned dataset for downstream modeling
df_to_modelling.to_parquet('../data/processed/telco_churn_clean.parquet')

# Export cleaned dataset for downstream analysis
df.to_parquet('../data/interim/telco_churn_clean.parquet')